# LangGraph Framework — Async Version with User Input Example

This is an **async, FastAPI-based** version of the dynamic LangGraph framework. It uses async OpenAI calls and supports plugin-style MCP servers that can be extended dynamically.

---

## Project Tree

```
langgraph/
├── README.md
├── requirements.txt
├── Dockerfile
├── .env.example
├── app/
│   ├── __init__.py
│   ├── main.py                  # FastAPI app + endpoints
│   ├── parent_agent.py          # ParentAgent async version
│   ├── mcp_registry.py          # Registry & decorator
│   ├── openai_client.py         # Async OpenAI helper
│   ├── servers/
│   │   ├── __init__.py
│   │   ├── weather.py
│   │   └── math.py
│   └── tools/
│       ├── __init__.py
│       ├── mock_weather.py
│       └── safe_eval.py
├── tests/
│   └── test_flow.py
```

---

## requirements.txt

```text
fastapi==0.115.0
uvicorn==0.22.0
openai==1.31.0
python-dotenv==1.0.0
pydantic==1.10.11
pytest==7.4.0
httpx==0.24.0
```

---

## .env.example

```text
OPENAI_API_KEY=sk-REPLACE_ME
OPENAI_MODEL=gpt-4o
LOG_LEVEL=info
```

---

## app/openai_client.py

```python
import os
import json
from openai import AsyncOpenAI

client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")


async def call_llm_chat(system: str, user_prompt: str, model: str = None, max_tokens: int = 400, temperature: float = 0.2) -> str:
    """Async call to OpenAI Chat Completion."""
    _model = model or MODEL
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_prompt},
    ]
    resp = await client.chat.completions.create(
        model=_model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()


def try_parse_json_block(text: str):
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            return None
    return None
```

---

## app/mcp_registry.py

```python
from typing import Callable, Dict

MCP_REGISTRY: Dict[str, Callable] = {}


def register_mcp(name: str):
    def decorator(factory):
        MCP_REGISTRY[name] = factory
        return factory
    return decorator
```

---

## app/parent_agent.py

```python
import json
from typing import Dict, Any
from .openai_client import call_llm_chat, try_parse_json_block
from .mcp_registry import MCP_REGISTRY


class ParentAgent:
    def __init__(self, system_prompt: str = "You are a routing agent."):
        self.system_prompt = system_prompt

    async def route(self, user_input: str) -> Dict[str, Any]:
        router_prompt = (
            "You are a router. Decide which MCP server should handle the user's input.\n"
            "Return JSON with keys: mcp (string or null), reasoning (string), mcp_payload (object).\n\n"
            "Available MCP options: " + ", ".join(MCP_REGISTRY.keys()) + "\n\n"
            f"User input: '''{user_input}'''"
        )

        llm_resp = await call_llm_chat(self.system_prompt, router_prompt)
        parsed = try_parse_json_block(llm_resp) or {"mcp": None, "reasoning": llm_resp, "mcp_payload": {}}

        mcp_name = parsed.get("mcp")
        reasoning = parsed.get("reasoning", "")
        payload = parsed.get("mcp_payload", {})

        result = {"decision": {"mcp": mcp_name, "reasoning": reasoning, "payload": payload}}

        if mcp_name and mcp_name in MCP_REGISTRY:
            client_factory = MCP_REGISTRY[mcp_name]
            client = client_factory()
            mcp_response = await client.call(payload.get("text", user_input), meta=payload)
            result["mcp_response"] = mcp_response
        else:
            direct_answer = await call_llm_chat("You are a helpful assistant.", user_input, max_tokens=200)
            result["direct_answer"] = direct_answer

        return result
```

---

## app/servers/weather.py

```python
from typing import Dict, Any
from ..mcp_registry import register_mcp
from ..openai_client import call_llm_chat, try_parse_json_block
from ..tools.mock_weather import get_mock_weather


@register_mcp("weather_mcp")
class WeatherMCPClient:
    def __init__(self):
        self.server = WeatherMCPServer()

    async def call(self, text: str, meta: Dict = None) -> Dict[str, Any]:
        return await self.server.handle(text, meta or {})


class WeatherMCPServer:
    async def handle(self, text: str, meta: Dict) -> Dict[str, Any]:
        prompt = (
            "You are the Weather MCP. Parse the user's text and return JSON with keys: {\n"
            "  \"tool\": \"<tool_name or null>\",\n"
            "  \"params\": {...},\n"
            "  \"reasoning\": \"...\"\n"
            "}\n\n"
            f"User text: '''{text}'''"
        )
        llm_resp = await call_llm_chat("Weather MCP agent.", prompt)
        parsed = try_parse_json_block(llm_resp) or {"tool": None, "params": {}, "reasoning": llm_resp}

        tool = parsed.get("tool")
        params = parsed.get("params", {})
        reasoning = parsed.get("reasoning", "")

        if tool == "mock_weather_api":
            city = params.get("city") or self._extract_city(text) or "Unknown"
            date = params.get("date") or "today"
            weather = get_mock_weather(city, date)
            return {"tool": tool, "params": {"city": city, "date": date}, "reasoning": reasoning, "result": weather}

        return {"tool": None, "reasoning": reasoning, "result": {"raw": text}}

    def _extract_city(self, text: str):
        words = text.split()
        for w in words:
            if w.istitle():
                return w
        return None
```

---

## app/servers/math.py

```python
from typing import Dict, Any
from ..mcp_registry import register_mcp
from ..openai_client import call_llm_chat, try_parse_json_block
from ..tools.safe_eval import safe_eval_expr


@register_mcp("math_mcp")
class MathMCPClient:
    def __init__(self):
        self.server = MathMCPServer()

    async def call(self, text: str, meta: Dict = None) -> Dict[str, Any]:
        return await self.server.handle(text, meta or {})


class MathMCPServer:
    async def handle(self, text: str, meta: Dict) -> Dict[str, Any]:
        prompt = (
            "You are the Math MCP. Return JSON: {\n"
            "  \"tool\": \"python_eval\" or \"none\",\n"
            "  \"expr\": \"<python expression>\",\n"
            "  \"reasoning\": \"...\"\n"
            "}\n\n"
            f"User text: '''{text}'''"
        )
        llm_resp = await call_llm_chat("Math MCP agent.", prompt)
        parsed = try_parse_json_block(llm_resp) or {"tool": None, "expr": None, "reasoning": llm_resp}

        tool = parsed.get("tool")
        expr = parsed.get("expr")
        reasoning = parsed.get("reasoning", "")

        if tool == "python_eval" and expr:
            result = safe_eval_expr(expr)
            return {"tool": tool, "expr": expr, "reasoning": reasoning, "result": result}

        return {"tool": None, "reasoning": reasoning, "result": "No operation"}
```

---

## app/tools/mock_weather.py

```python
def get_mock_weather(city: str, date: str):
    return {"city": city, "date": date, "forecast": "Sunny", "temp_c": 28}
```

---

## app/tools/safe_eval.py

```python
import ast

def safe_eval_expr(expr: str):
    try:
        node = ast.parse(expr, mode="eval")
    except Exception as e:
        return {"error": str(e)}
    for n in ast.walk(node):
        if not isinstance(n, (ast.Expression, ast.BinOp, ast.Num, ast.Constant, ast.Add, ast.Sub, ast.Mult, ast.Div)):
            return {"error": f"Disallowed node: {type(n).__name__}"}
    try:
        result = eval(compile(node, filename="<safe>", mode="eval"), {"__builtins__": {}}, {})
        return {"value": result}
    except Exception as e:
        return {"error": str(e)}
```

---

## app/main.py

```python
from fastapi import FastAPI
from pydantic import BaseModel
from .parent_agent import ParentAgent

app = FastAPI(title="LangGraph Async API")
parent = ParentAgent()

class Query(BaseModel):
    text: str

@app.post("/route")
async def route(query: Query):
    return await parent.route(query.text)

@app.get("/health")
async def health():
    return {"status": "ok"}
```

---

## User Input Example (Run in Python shell)

```python
import asyncio
from app.parent_agent import ParentAgent

async def main():
    parent = ParentAgent()
    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit"]:
            break
        response = await parent.route(user_input)
        print("\nAI Response:")
        print(response)

if __name__ == "__main__":
    asyncio.run(main())
```

---

## Dockerfile

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt
COPY . /app
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

---

## README.md

````md
# Async LangGraph Framework

### Run locally

```bash
cp .env.example .env
pip install -r requirements.txt
uvicorn app.main:app --reload --port 8000
````

### Example query

```bash
curl -X POST http://localhost:8000/route -H "Content-Type: application/json" -d '{"text":"What's the weather in Bengaluru tomorrow?"}'
```

### Interactive mode

```bash
python app/interactive.py
```

### Extend

Add new MCP in `app/servers/` and register using `@register_mcp("your_mcp")`.

```

---

This version supports **async LLM calls**, **FastAPI endpoints**, and **interactive CLI usage**.

```
